<a href="https://colab.research.google.com/github/zhangling297/tensor2tensor/blob/master/Copy_of_INBreast_Cancer_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Report: TransUNet and Pixel-Level Contrastive Learning for Medical Image Segmentation

## 1. What the Topic Is
The topic explores the integration of **TransUNet** (a hybrid Vision Transformer and U-Net architecture) combined with **Pixel-Level Contrastive Learning** and **Focal Loss** to perform precise medical image segmentation. Specifically, this approach is applied to the INbreast dataset to segment breast cancer lesions, which often present with fuzzy and ill-defined boundaries.

## 2. What Problem Was It Designed to Address
Medical image segmentation, particularly for tumors in mammography, faces several critical challenges:
1. **Fuzzy Boundaries:** Low contrast between dense breast tissue and tumors makes it difficult for standard Convolutional Neural Networks (CNNs) to accurately delineate boundaries because they rely heavily on local pixel neighborhoods (limited receptive fields).
2. **Lack of Global Context:** Standard U-Nets struggle to understand the global anatomical structure of the breast, which is crucial for distinguishing a benign density from a malignant mass.
3. **Severe Class Imbalance:** In a typical mammogram, tumor pixels represent a tiny fraction (< 5%) of the total image, while the background and healthy tissue dominate. This biases traditional loss functions (like standard Cross-Entropy) toward predicting only the background.

## 3. Who Introduced It, When, and Where
The solution is a culmination of several pivotal breakthroughs in deep learning:
*   **U-Net:** Introduced by **Olaf Ronneberger, Philipp Fischer, and Thomas Brox** in **2015** at the **MICCAI** conference. It established the standard encoder-decoder architecture with skip connections for biomedical image segmentation.
*   **Vision Transformer (ViT):** Introduced by **Alexey Dosovitskiy et al. (Google Brain)** at **ICLR 2021**. It demonstrated that treating an image as a sequence of patches and processing them with pure self-attention (Transformers) captures excellent global context.
*   **TransUNet:** Introduced by **Jieneng Chen et al.** in **2021** (arXiv/CVPR). It bridged ViT and U-Net, using a CNN-Transformer hybrid encoder to capture both high-resolution local features and rich global context.
*   **Contrastive Learning (SimCLR / InfoNCE):** Popularized by **Ting Chen et al. (Google Brain)** at **ICML 2020**. It established the framework for learning representations by maximizing agreement between differently augmented views of the same data point via the NT-Xent loss.

## 4. How It Works (Theoretical & Mathematical Foundation)

### A. TransUNet Architecture
TransUNet first passes the image through a CNN feature extractor to get a feature map. This map is then divided into patches, flattened, and projected into sequence embeddings. These sequence embeddings are fed into a standard Transformer Encoder (comprising Multi-Head Self-Attention layers), which models long-range dependencies. Finally, the sequence is reshaped back into a spatial feature map and passed to a cascaded U-Net decoder, which progressively upsamples the features while fusing them with high-resolution CNN features via skip connections.

### B. Contrastive Learning & NT-Xent Loss
Contrastive learning forces the model to learn features such that similar samples (positive pairs) are close in the latent space, and dissimilar samples (negative pairs) are far apart.
The **Normalized Temperature-scaled Cross Entropy (NT-Xent) Loss** for a positive pair $(z_i, z_j)$ is defined as:

$$ \ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)} $$

Where:
*   $\text{sim}(u, v) = \frac{u^T v}{\|u\| \|v\|}$ is the cosine similarity.
*   $\tau$ is a temperature parameter that scales the distribution.
*   $N$ is the batch size (so there are $2N$ augmented views, providing $2N-1$ negatives).

In our *pixel-level* adaptation, instead of pulling whole images together, we pull dense feature representations of the *same spatial pixel* from two augmented views together, explicitly forcing the model to learn robust tissue boundary features invariant to spatial distortions.

### C. Focal Loss for Imbalance
To address the extreme background-to-tumor imbalance, we replaced standard Cross-Entropy with **Focal Loss** (Lin et al., ICCV 2017). It dynamically scales the cross-entropy loss based on prediction confidence:

$$ \text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t) $$

Where $\gamma$ (focusing parameter) reduces the relative loss for well-classified examples (easy background pixels), putting more focus on hard, misclassified examples (tumor boundaries).

## 5. Implementation in PyTorch
Below are the core PyTorch implementations utilized in our experimental setup.

**1. The NT-Xent Contrastive Loss:**
```python
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        z = torch.cat((z_i, z_j), dim=0)
        sim_matrix = torch.matmul(z, z.T) / self.temperature
        
        labels = torch.cat([torch.arange(batch_size) + batch_size, torch.arange(batch_size)], dim=0).to(z.device)
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
        sim_matrix.masked_fill_(mask, -9e15)
        
        return self.criterion(sim_matrix, labels)
```

**2. Focal + Dice Loss:**
```python
class FocalDiceLoss(nn.Module):
    def __init__(self, weight_focal=1.0, weight_dice=2.0, gamma=3.0):
        super(FocalDiceLoss, self).__init__()
        self.focal = FocalLoss(gamma=gamma) # Custom FocalLoss implementation
        self.weight_focal = weight_focal
        self.weight_dice = weight_dice

    def forward(self, inputs, targets):
        f_loss = self.focal(inputs, targets)
        probs = F.softmax(inputs, dim=1)[:, 1, :, :]
        probs_flat = probs.contiguous().view(-1)
        targets_flat = targets.contiguous().view(-1).float()
        
        intersection = (probs_flat * targets_flat).sum()
        d_loss = 1 - ((2. * intersection + 1e-6) / (probs_flat.sum() + targets_flat.sum() + 1e-6))
        return self.weight_focal * f_loss + self.weight_dice * d_loss
```

## 6. Experimental Setup and Results
Our experiment utilized the INbreast dataset, preprocessing full-resolution mammograms into 256x256 tensors. We executed two primary fine-tuning phases on our TransUNet model:

1.  **Phase 1 (Dice + CrossEntropy):** Over 100 epochs, the model achieved a best Validation Dice score of ~0.4285. However, quantitative evaluation revealed that standard CE loss favored predicting background, resulting in a staggering 12,653 False Negatives (missed tumor pixels) versus only 598 True Positives.
2.  **Phase 2 (Focal + Dice Loss):** By re-initializing the model and applying a highly weighted Focal Loss ($\gamma=3.0$), we explicitly penalized the model for missing difficult tumor pixels. Over 50 epochs, this shifted the learning dynamics. The quantitative results showed an increase in True Positives (1,462) and low False Positives (300). The False Negatives dropped to 11,789.

**Conclusion:** While Focal Loss successfully forced the network to identify more tumor structure (evidenced by the >2x increase in True Positives), the absolute number of False Negatives remains high, resulting in an overall Dice of ~0.1948 in the final evaluation. This highlights that while Contrastive Learning and TransUNet provide powerful theoretical advantages for context, severe medical image imbalance requires highly nuanced thresholding, patching strategies, or cascaded detection mechanisms to fully eradicate False Negatives.

# Report: TransUNet and Pixel-Level Contrastive Learning for Medical Image Segmentation

## 1. What the Topic Is
The topic explores the integration of **TransUNet** (a hybrid Vision Transformer and U-Net architecture) combined with **Pixel-Level Contrastive Learning** and **Focal Loss** to perform precise medical image segmentation. Specifically, this approach is applied to the INbreast dataset to segment breast cancer lesions, which often present with fuzzy and ill-defined boundaries.

## 2. What Problem Was It Designed to Address
Medical image segmentation, particularly for tumors in mammography, faces several critical challenges:
1. **Fuzzy Boundaries:** Low contrast between dense breast tissue and tumors makes it difficult for standard Convolutional Neural Networks (CNNs) to accurately delineate boundaries because they rely heavily on local pixel neighborhoods (limited receptive fields).
2. **Lack of Global Context:** Standard U-Nets struggle to understand the global anatomical structure of the breast, which is crucial for distinguishing a benign density from a malignant mass.
3. **Severe Class Imbalance:** In a typical mammogram, tumor pixels represent a tiny fraction (< 5%) of the total image, while the background and healthy tissue dominate. This biases traditional loss functions (like standard Cross-Entropy) toward predicting only the background.

## 3. Who Introduced It, When, and Where
The solution is a culmination of several pivotal breakthroughs in deep learning:
*   **U-Net:** Introduced by **Olaf Ronneberger, Philipp Fischer, and Thomas Brox** in **2015** at the **MICCAI** conference. It established the standard encoder-decoder architecture with skip connections for biomedical image segmentation.
*   **Vision Transformer (ViT):** Introduced by **Alexey Dosovitskiy et al. (Google Brain)** at **ICLR 2021**. It demonstrated that treating an image as a sequence of patches and processing them with pure self-attention (Transformers) captures excellent global context.
*   **TransUNet:** Introduced by **Jieneng Chen et al.** in **2021** (arXiv/CVPR). It bridged ViT and U-Net, using a CNN-Transformer hybrid encoder to capture both high-resolution local features and rich global context.
*   **Contrastive Learning (SimCLR / InfoNCE):** Popularized by **Ting Chen et al. (Google Brain)** at **ICML 2020**. It established the framework for learning representations by maximizing agreement between differently augmented views of the same data point via the NT-Xent loss.

## 4. How It Works (Theoretical & Mathematical Foundation)

### A. TransUNet Architecture
TransUNet first passes the image through a CNN feature extractor to get a feature map. This map is then divided into patches, flattened, and projected into sequence embeddings. These sequence embeddings are fed into a standard Transformer Encoder (comprising Multi-Head Self-Attention layers), which models long-range dependencies. Finally, the sequence is reshaped back into a spatial feature map and passed to a cascaded U-Net decoder, which progressively upsamples the features while fusing them with high-resolution CNN features via skip connections.

### B. Contrastive Learning & NT-Xent Loss
Contrastive learning forces the model to learn features such that similar samples (positive pairs) are close in the latent space, and dissimilar samples (negative pairs) are far apart.
The **Normalized Temperature-scaled Cross Entropy (NT-Xent) Loss** for a positive pair $(z_i, z_j)$ is defined as:

$$ \ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)} $$

Where:
*   $\text{sim}(u, v) = \frac{u^T v}{\|u\| \|v\|}$ is the cosine similarity.
*   $\tau$ is a temperature parameter that scales the distribution.
*   $N$ is the batch size (so there are $2N$ augmented views, providing $2N-1$ negatives).

In our *pixel-level* adaptation, instead of pulling whole images together, we pull dense feature representations of the *same spatial pixel* from two augmented views together, explicitly forcing the model to learn robust tissue boundary features invariant to spatial distortions.

### C. Focal Loss for Imbalance
To address the extreme background-to-tumor imbalance, we replaced standard Cross-Entropy with **Focal Loss** (Lin et al., ICCV 2017). It dynamically scales the cross-entropy loss based on prediction confidence:

$$ \text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t) $$

Where $\gamma$ (focusing parameter) reduces the relative loss for well-classified examples (easy background pixels), putting more focus on hard, misclassified examples (tumor boundaries).

## 5. Implementation in PyTorch
Below are the core PyTorch implementations utilized in our experimental setup.

**1. The NT-Xent Contrastive Loss:**
```python
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        z = torch.cat((z_i, z_j), dim=0)
        sim_matrix = torch.matmul(z, z.T) / self.temperature
        
        labels = torch.cat([torch.arange(batch_size) + batch_size, torch.arange(batch_size)], dim=0).to(z.device)
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
        sim_matrix.masked_fill_(mask, -9e15)
        
        return self.criterion(sim_matrix, labels)
```

**2. Focal + Dice Loss:**
```python
class FocalDiceLoss(nn.Module):
    def __init__(self, weight_focal=1.0, weight_dice=2.0, gamma=3.0):
        super(FocalDiceLoss, self).__init__()
        self.focal = FocalLoss(gamma=gamma) # Custom FocalLoss implementation
        self.weight_focal = weight_focal
        self.weight_dice = weight_dice

    def forward(self, inputs, targets):
        f_loss = self.focal(inputs, targets)
        probs = F.softmax(inputs, dim=1)[:, 1, :, :]
        probs_flat = probs.contiguous().view(-1)
        targets_flat = targets.contiguous().view(-1).float()
        
        intersection = (probs_flat * targets_flat).sum()
        d_loss = 1 - ((2. * intersection + 1e-6) / (probs_flat.sum() + targets_flat.sum() + 1e-6))
        return self.weight_focal * f_loss + self.weight_dice * d_loss
```

## 6. Experimental Setup and Results
Our experiment utilized the INbreast dataset, preprocessing full-resolution mammograms into 256x256 tensors. We executed two primary fine-tuning phases on our TransUNet model:

1.  **Phase 1 (Dice + CrossEntropy):** Over 100 epochs, the model achieved a best Validation Dice score of ~0.4285. However, quantitative evaluation revealed that standard CE loss favored predicting background, resulting in a staggering 12,653 False Negatives (missed tumor pixels) versus only 598 True Positives.
2.  **Phase 2 (Focal + Dice Loss):** By re-initializing the model and applying a highly weighted Focal Loss ($\gamma=3.0$), we explicitly penalized the model for missing difficult tumor pixels. Over 50 epochs, this shifted the learning dynamics. The quantitative results showed an increase in True Positives (1,462) and low False Positives (300). The False Negatives dropped to 11,789.

**Conclusion:** While Focal Loss successfully forced the network to identify more tumor structure (evidenced by the >2x increase in True Positives), the absolute number of False Negatives remains high, resulting in an overall Dice of ~0.1948 in the final evaluation. This highlights that while Contrastive Learning and TransUNet provide powerful theoretical advantages for context, severe medical image imbalance requires highly nuanced thresholding, patching strategies, or cascaded detection mechanisms to fully eradicate False Negatives.

### Developing Contrastive Learning

1. **Data Augmentation**:  Generate two different augmented views of the same image. (color jitter, random crop, Gaussian blur) .
2. **Encoder**: ResNet-50 used to extract features.
3. **Projection Head**: An MLP that maps the encoder representations to the space where the contrastive loss is applied.
4. **Loss Function**: NT-Xent. It treats the two augmented views of the same image as a positive pair and all other images in the batch as negatives.

Below is an application for a PyTorch-based contrastive learning setup.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# 1. Define the Model with a Projection Head
class ContrastiveModel(nn.Module):
    def __init__(self, base_model='resnet18', out_dim=128):
        super(ContrastiveModel, self).__init__()
        # Use a pretrained or untrained ResNet
        self.encoder = models.__dict__[base_model](weights=None)

        # Remove the classification head
        dim_mlp = self.encoder.fc.in_features
        self.encoder.fc = nn.Identity()

        # Add a projection head (MLP)
        self.projector = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, out_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1) # Normalize the outputs

model = ContrastiveModel()
print("Model initialized!")

In [ ]:
# 2. Define the Contrastive Loss (NT-Xent / InfoNCE)
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]

        # Concatenate the representations
        z = torch.cat((z_i, z_j), dim=0)

        # Compute similarity matrix
        sim_matrix = torch.matmul(z, z.T) / self.temperature

        # Create labels (each sample's positive pair is batch_size away)
        labels = torch.cat([torch.arange(batch_size) + batch_size, torch.arange(batch_size)], dim=0)
        labels = labels.to(z.device)

        # Mask out self-similarity
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
        sim_matrix.masked_fill_(mask, -9e15)

        loss = self.criterion(sim_matrix, labels)
        return loss

loss_fn = NTXentLoss()
print("Loss function initialized!")

### Next Steps:
To adapt this to your specific dataset:
1. Update your `Dataset` class to return *two* augmented versions of each input.
2. Write a training loop that passes both augmented batches (`x_i`, `x_j`) through the `ContrastiveModel` to get `z_i` and `z_j`.
3. Compute the `NTXentLoss(z_i, z_j)` and backpropagate.

## INbreast Tumor Segmentation with U-Net, ViT, and Contrastive Learning

Project idea addresses a highly relevant clinical problem: **fuzzy boundaries in breast cancer lesions** (benign vs. malignant) using the **INbreast dataset**.

### 1. Historical Context (Who, When, Where)
*   **INbreast Dataset**: Introduced by Inès C. Moreira et al. in **2012** (published in *Academic Radiology*). It's a widely used full-field digital mammography dataset with precise annotations (contours) for masses and calcifications.
*   **U-Net**: Introduced by **Olaf Ronneberger, Philipp Fischer, and Thomas Brox** in **2015** at the **MICCAI** (Medical Image Computing and Computer Assisted Intervention) conference. It became the gold standard for medical image segmentation due to its symmetric encoder-decoder structure and skip connections.
*   **Vision Transformer (ViT)**: Introduced by **Alexey Dosovitskiy et al. (Google Brain)** in late **2020** (published at **ICLR 2021**). It proved that pure transformers applied directly to sequences of image patches can perform exceedingly well on image classification.
*   ***Recommendation - TransUNet***: Instead of just basic ViT, look into **TransUNet** (Chen et al., 2021). It combines the U-Net architecture with Transformers, using ViT as the encoder to capture global context (great for fuzzy boundaries) and U-Net's decoder for precise localization.

### 2. Why this is a good idea
Fuzzy boundaries happen because the contrast between dense breast tissue and tumors is low. Standard CNNs (like basic U-Net) struggle because their receptive field is limited; they only look at local pixel neighborhoods.
**Transformers (ViT)** look at the global context (how the tumor relates to the whole breast tissue structure) using self-attention, making them much better at inferring where a fuzzy boundary *should* be based on global patterns.

### 3. How to implement Contrastive Learning for this task
Standard contrastive learning (like the SimCLR skeleton we built above) works at the **image level**. Target **segmentation** (pixel-level classification).

**Approach A: Encoder Pre-training (Standard)**
1. Extract ROIs (Regions of Interest) from the INbreast dataset.
2. Apply strong augmentations (blurring, rotations, elastic deformations) to create positive pairs.
3. Pre-train your U-Net or ViT **encoder** using the `ContrastiveModel` and `NTXentLoss` we defined earlier. The model learns to understand breast tissue textures without needing the segmentation masks.
4. Fine-tune the whole model (Encoder + Decoder) using standard segmentation losses (like Dice Loss or Cross Entropy) on the ground truth boundaries.

**Approach B: Dense / Pixel-Level Contrastive Learning (Advanced & Highly Recommended)**
Pull *pixels* of the same class together in the feature space.
1. Pass two augmented views of an image through the network.
2. Instead of a single vector `z`, output a dense feature map.
3. Apply **Pixel-to-Pixel Contrastive Loss**: Take a feature vector for a "Tumor Boundary" pixel in View 1, and make it attract the corresponding "Tumor Boundary" pixel in View 2, while pushing it away from "Background" pixels or "Benign" pixels.
4. **Why this helps:** By forcing the model to explicitly push "tumor boundary" features far away from "healthy tissue" features in the latent space, the model naturally learns a sharper boundary, combating the fuzziness of the mammogram.

### Pipeline:
1. Preprocess INbreast (extract ROIs, normalize).
2. Set up a Hybrid Model (e.g., TransUNet).
3. Apply Pixel-level Contrastive Learning as an auxiliary loss alongside your main Dice Loss during training.

### Alternative:


In [ ]:
import json
import os

# Kaggle auth (SAFE): store credentials in environment variables.
# In Google Colab: Runtime → Secrets → add KAGGLE_USERNAME and KAGGLE_KEY.

kaggle_username = os.environ.get("KAGGLE_USERNAME")
kaggle_key = os.environ.get("KAGGLE_KEY")

if not kaggle_username or not kaggle_key:
    raise RuntimeError(
        "Missing Kaggle credentials. Set env vars KAGGLE_USERNAME and KAGGLE_KEY (Colab Secrets recommended)."
    )

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials configured from environment variables.")


In [ ]:
!kaggle datasets download -d tommyngx/inbreast2012 -p /content/inbreast2012

In [ ]:
!kaggle datasets download -d martholi/inbreast -p /content/inbreast

In [ ]:
from google.colab import drive

# log in and grant access to your Google Drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil

# Define the destination folder in your Google Drive
drive_folder = '/content/drive/MyDrive/INbreast_Project'
os.makedirs(drive_folder, exist_ok=True)
print(f"Created directory: {drive_folder}")

# Paths to the downloaded zip files
zip1 = '/content/inbreast2012/inbreast2012.zip'
zip2 = '/content/inbreast/inbreast.zip'

# Copy the files to Google Drive
if os.path.exists(zip1):
    print(f"Copying {zip1} to Drive...")
    shutil.copy(zip1, drive_folder)
    print("Done!")

if os.path.exists(zip2):
    print(f"Copying {zip2} to Drive...")
    shutil.copy(zip2, drive_folder)
    print("Done!")

print(f"\nSuccessfully saved the datasets to {drive_folder}!")

In [ ]:
import zipfile
import os
import shutil
import glob
import tarfile
from google.colab import drive

# Mount Google Drive to access the data
drive.mount('/content/drive')

# Define paths for the downloaded zip files located in Google Drive
zip_path_inbreast2012 = '/content/drive/MyDrive/INbreast_Project/inbreast2012.zip'
zip_path_inbreast = '/content/drive/MyDrive/INbreast_Project/inbreast.zip'

# Define extraction directories
dcm_extract_dir = '/content/inbreast_dcm'
xml_extract_dir = '/content/inbreast_xml'

os.makedirs(dcm_extract_dir, exist_ok=True)
os.makedirs(xml_extract_dir, exist_ok=True)

print(f"Checking if {zip_path_inbreast2012} exists: {os.path.exists(zip_path_inbreast2012)}")
if os.path.exists(zip_path_inbreast2012):
    print("Extracting INbreast2012.zip...")
    with zipfile.ZipFile(zip_path_inbreast2012, 'r') as zip_ref:
        for member in zip_ref.namelist():
            # Extract DICOM files
            if ('AllDICOMs' in member or 'ALL-IMGS' in member) and member.lower().endswith('.dcm'):
                source = zip_ref.open(member)
                target = open(os.path.join(dcm_extract_dir, os.path.basename(member)), 'wb')
                with source, target:
                    shutil.copyfileobj(source, target)
            # Extract XML files
            elif 'AllXML' in member and member.lower().endswith('.xml'):
                source = zip_ref.open(member)
                target = open(os.path.join(xml_extract_dir, os.path.basename(member)), 'wb')
                with source, target:
                    shutil.copyfileobj(source, target)
    print("Extraction of INbreast2012.zip complete.")
else:
    print(f"Error: Could not find {zip_path_inbreast2012}.")

print(f"Checking if {zip_path_inbreast} exists: {os.path.exists(zip_path_inbreast)}")
if os.path.exists(zip_path_inbreast):
    print("Extracting inbreast.zip...")
    with zipfile.ZipFile(zip_path_inbreast, 'r') as zip_ref:
        zip_ref.extract('inbreast.tgz', '/content/')

    print("Extracting inbreast.tgz...")
    with tarfile.open('/content/inbreast.tgz', 'r:gz') as tar_ref:
        for member in tar_ref.getmembers():
            if member.isfile():
                if ('ALL-IMGS' in member.name or 'AllDICOMs' in member.name) and member.name.lower().endswith('.dcm'):
                    f = tar_ref.extractfile(member)
                    if f is not None:
                        with open(os.path.join(dcm_extract_dir, os.path.basename(member.name)), 'wb') as target:
                            shutil.copyfileobj(f, target)
                elif 'AllXML' in member.name and member.name.lower().endswith('.xml'):
                    f = tar_ref.extractfile(member)
                    if f is not None:
                        with open(os.path.join(xml_extract_dir, os.path.basename(member.name)), 'wb') as target:
                            shutil.copyfileobj(f, target)
    print("Extraction of inbreast.zip complete.")
else:
    print(f"Error: Could not find {zip_path_inbreast}.")

# Update dcm_dir and xml_dir for subsequent cells
dcm_dir = dcm_extract_dir
xml_dir = xml_extract_dir

# Get the list of DICOM and XML files
dcm_files = glob.glob(os.path.join(dcm_dir, '*.dcm'))
xml_files = glob.glob(os.path.join(xml_dir, '*.xml'))

print(f"Found {len(dcm_files)} DICOM files in {dcm_dir}")
print(f"Found {len(xml_files)} XML files in {xml_dir}")

if not dcm_files:
    print("Warning: No DICOM files found. Please check extraction paths and contents.")
if not xml_files:
    print("Warning: No XML files found. Annotations might be missing.")

In [ ]:
import zipfile

zip_path_inbreast2012 = '/content/drive/MyDrive/INbreast_Project/inbreast2012.zip'
print("Searching for XML files in INbreast2012.zip...")
with zipfile.ZipFile(zip_path_inbreast2012, 'r') as zip_ref:
    xml_files_in_zip = [m for m in zip_ref.namelist() if m.endswith('.xml')]
    print(f"Found {len(xml_files_in_zip)} XML files.")
    if xml_files_in_zip:
        print("Sample XML paths:")
        for path in xml_files_in_zip[:10]:
            print(path)


In [ ]:
!pip install pydicom -q

In [ ]:
!pip install pydicom -q
import pydicom
import matplotlib.pyplot as plt
import os
import glob

# Path to the directory where the DICOM files are stored
dcm_dir = '/content/inbreast_dcm'

# Find all DICOM files in the directory
dcm_files = glob.glob(os.path.join(dcm_dir, '*.dcm'))

if dcm_files:
    # Pick the first file as our sample
    sample_file = dcm_files[0]
    print(f"Loading sample image: {os.path.basename(sample_file)}")

    # Read the DICOM file using pydicom
    dicom_data = pydicom.dcmread(sample_file)

    # Extract the pixel array (the actual image)
    image_pixels = dicom_data.pixel_array

    # Display the image using matplotlib
    plt.figure(figsize=(8, 10))
    plt.imshow(image_pixels, cmap='gray') # Display in grayscale
    plt.title(f"INbreast Sample\nShape: {image_pixels.shape}")
    plt.axis('off') # Hide axes for a cleaner look
    plt.show()
else:
    print("Could not find any DICOM files. Please check if the download completed successfully.")

Now that the data is extracted and paths are set, the next step is to use the `INbreastDenseContrastiveDataset` to prepare data for pixel-level contrastive pre-training. I'll modify the existing pre-training loop to use this dataset and save the pre-trained encoder.

### 1. Pixel-Level Contrastive Loss
This loss function takes the dense feature maps from the two augmented views and pulls features from the same spatial location (positive pairs) together, while pushing them away from features in other locations or classes (negative pairs).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PixelContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(PixelContrastiveLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j, mask):
        # z_i, z_j shape: (Batch, Channels, Height, Width)
        B, C, H, W = z_i.shape

        # Flatten spatial dimensions -> (Batch, Channels, H*W)
        z_i_flat = z_i.view(B, C, -1)
        z_j_flat = z_j.view(B, C, -1)

        # Rearrange to (Batch * H * W, Channels) to treat each pixel as a sample
        z_i_flat = z_i_flat.permute(0, 2, 1).contiguous().view(-1, C)
        z_j_flat = z_j_flat.permute(0, 2, 1).contiguous().view(-1, C)

        # For simplicity in this skeleton, we treat corresponding spatial pixels as positive pairs.
        # In a full implementation, you would use `mask` to sample specific classes (like boundaries)
        # and only compute loss on those sampled pixels to save memory.

        # Number of total pixels
        N = z_i_flat.shape[0]

        # Similarity matrix (N x N)
        # Note: In practice, N is too large for full matrix multiplication (e.g., 256x256 = 65536 pixels).
        # You must sample a subset of pixels (e.g., 1024 random pixels or boundary pixels) per batch.
        # Here we simulate with a small subset to prevent OOM errors.

        sample_size = min(N, 1024)
        indices = torch.randperm(N)[:sample_size]

        z_i_sub = z_i_flat[indices]
        z_j_sub = z_j_flat[indices]

        # Calculate InfoNCE on the subset
        z_both = torch.cat([z_i_sub, z_j_sub], dim=0)
        sim_matrix = torch.matmul(z_both, z_both.T) / self.temperature

        labels = torch.cat([torch.arange(sample_size) + sample_size, torch.arange(sample_size)], dim=0).to(z_i.device)

        # Mask out self-similarity
        mask_self = torch.eye(2 * sample_size, dtype=torch.bool).to(z_i.device)
        sim_matrix.masked_fill_(mask_self, -9e15)

        loss = self.criterion(sim_matrix, labels)
        return loss

pixel_loss_fn = PixelContrastiveLoss()
print("Pixel-Level Contrastive Loss initialized!")

### 2. INbreast Dataset Preprocessing & Augmentations
For contrastive learning, dataset returns *two* augmented versions of the same image. use `torchvision.transforms` (or `albumentations` for more advanced spatial+mask augmentations) to apply random crops, flips, and color jitters.

### 6. TransUNet Decoder
The decoder's job is to take the dense, context-rich feature map from the ViT bottleneck (e.g., `32x32`) and progressively upsample it back to the original image resolution (e.g., `256x256`), outputting the final class probabilities for each pixel.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransUNetDecoder(nn.Module):
    def __init__(self, in_channels=512, num_classes=2):
        super(TransUNetDecoder, self).__init__()

        # Upsampling block 1: 32x32 -> 64x64
        self.up1 = nn.ConvTranspose2d(in_channels, 256, kernel_size=2, stride=2)
        self.conv1 = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # Upsampling block 2: 64x64 -> 128x128
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Upsampling block 3: 128x128 -> 256x256
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        # Final output layer
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # x shape from encoder: (B, 512, 32, 32)
        x = self.up1(x)
        x = self.conv1(x)

        x = self.up2(x)
        x = self.conv2(x)

        x = self.up3(x)
        x = self.conv3(x)

        out = self.final_conv(x)
        return out

# Test the Decoder
decoder = TransUNetDecoder(in_channels=512, num_classes=2)
dummy_encoded = torch.randn(2, 512, 32, 32)
decoder_out = decoder(dummy_encoded)

print(f"Encoder output shape (input to decoder): {dummy_encoded.shape}")
print(f"Final Segmentation Map shape: {decoder_out.shape}")
print("Decoder initialized successfully!")

### 7. Training Loop for Dense Contrastive Pre-training
This loop focuses *only* on the contrastive pre-training phase (Approach B).  Use the `TransUNetEncoder` and the `DenseContrastiveModel` projector. The goal is to learn powerful, boundary-aware feature representations before fine-tuning the full segmentation model.

### 8. Save Processed Dataset for Later
 Iterate through dataset, extract the raw images and masks, and save them directly as PyTorch tensors `.pt` files to Google Drive.

In [ ]:
import os
import torch
from tqdm import tqdm

# Directory to save the processed tensors
save_dir = '/content/drive/MyDrive/INbreast_Processed'
os.makedirs(save_dir, exist_ok=True)

# Using the correct xml_dir defined earlier during extraction
xml_dir = '/content/inbreast_xml'
os.makedirs(xml_dir, exist_ok=True)

print(f"Saving processed dataset to {save_dir}...")

# Process ALL files by passing the entire dcm_files list
# We use the dataset class we created earlier
dataset = INbreastDenseContrastiveDataset(dcm_files=dcm_files, xml_dir=xml_dir, image_size=256)

for i in tqdm(range(len(dataset))):
    # For fine-tuning, we only strictly need the image and the mask.
    # The dataset returns augmented views, but for the saved base dataset,
    # you would ideally save the unaugmented base tensors.
    # We'll just grab the first view (which has spatial transforms but is fine for this skeleton)
    v1_img, v1_mask, _, _ = dataset[i]

    # Save as dictionary
    data_dict = {
        'image': v1_img,
        'mask': v1_mask
    }
    torch.save(data_dict, os.path.join(save_dir, f'sample_{i}.pt'))

print("Dataset successfully processed and saved!")

The previous cell saved the *unaugmented* image and mask pairs. Now, we will load these processed files using `ProcessedINbreastDataset` and prepare `DataLoader` instances for the fine-tuning phase. After that, we can run the full fine-tuning process.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os

# Ensure device is set
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- 1. Combine Encoder and Decoder into TransUNet ---
class TransUNetDecoder(nn.Module):
    def __init__(self, in_channels=512, num_classes=2):
        super(TransUNetDecoder, self).__init__()
        self.up1 = nn.ConvTranspose2d(in_channels, 256, kernel_size=2, stride=2)
        self.conv1 = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.up1(x)
        x = self.conv1(x)
        x = self.up2(x)
        x = self.conv2(x)
        x = self.up3(x)
        x = self.conv3(x)
        out = self.final_conv(x)
        return out

class TransUNet(nn.Module):
    def __init__(self, encoder, decoder):
        super(TransUNet, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        features = self.encoder(x)
        out = self.decoder(features)
        return out

# Initialize the Encoder (using the class we redefined recently)
encoder = TransUNetEncoder(in_channels=1, hidden_dim=512).to(device)

# Try to load pre-trained encoder weights
pretrained_encoder_dir = '/content/drive/MyDrive/INbreast_Project/Pretrained_Encoder'
encoder_checkpoint_path = os.path.join(pretrained_encoder_dir, 'transunet_encoder_pretrained.pth')

if os.path.exists(encoder_checkpoint_path):
    try:
        encoder.load_state_dict(torch.load(encoder_checkpoint_path, map_location=device))
        print(f"Successfully loaded pre-trained encoder weights from {encoder_checkpoint_path}")
    except Exception as e:
        print(f"Could not load pre-trained weights due to: {e}. Training encoder from scratch.")
else:
    print(f"Pre-trained encoder weights not found at {encoder_checkpoint_path}. Training encoder from scratch.")

# Initialize the Decoder
decoder = TransUNetDecoder(in_channels=512, num_classes=2).to(device)

# Initialize the Full Model
full_model = TransUNet(encoder, decoder).to(device)
print("Full TransUNet model initialized!")

# --- 2. Fine-Tuning Setup ---
# Unfreeze all layers
for param in full_model.parameters():
    param.requires_grad = True
print("All layers of TransUNet are unfrozen for fine-tuning.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(full_model.parameters(), lr=1e-4)
print("Optimizer and Loss Function ready for fine-tuning.")


In [ ]:
import glob
import torch
import os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

class ProcessedINbreastDataset(Dataset):
    def __init__(self, data_dir):
        self.file_paths = glob.glob(os.path.join(data_dir, '*.pt'))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load the saved dictionary
        data = torch.load(self.file_paths[idx])

        # The image and mask are (1, H, W)
        img = data['image']
        mask = data['mask']

        # Force resize to (256, 256) to fix dimension mismatches during batching
        img = F.interpolate(img.unsqueeze(0), size=(256, 256), mode='bilinear', align_corners=False).squeeze(0)
        mask = F.interpolate(mask.unsqueeze(0), size=(256, 256), mode='nearest').squeeze(0).squeeze(0).long()

        return img, mask

# 1. Initialize the dataset
processed_dir = '/content/drive/MyDrive/INbreast_Processed'
full_dataset = ProcessedINbreastDataset(processed_dir)

# Ensure we have data
if len(full_dataset) == 0:
    print(f"Warning: No processed files found in {processed_dir}. Please check step 8.")
else:
    print(f"Total processed samples found: {len(full_dataset)}")

    # 2. Split into Train (80%) and Validation (20%)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    # Use a fixed generator seed for reproducible splits
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

    # 3. Create DataLoaders
    batch_size = 4
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f"Training set: {len(train_dataset)} samples ({len(train_loader)} batches)")
    print(f"Validation set: {len(val_dataset)} samples ({len(val_loader)} batches)")
    print("DataLoaders are ready for training!")


### 13. Create Train and Validation DataLoaders
Now we load the preprocessed dataset from Google Drive, split it into training and validation sets, and create `DataLoader` instances to feed the data into our model in batches.

In [ ]:
import glob
import torch
import os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

class ProcessedINbreastDataset(Dataset):
    def __init__(self, data_dir):
        self.file_paths = glob.glob(os.path.join(data_dir, '*.pt'))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load the saved dictionary
        data = torch.load(self.file_paths[idx])

        # The image and mask are (1, H, W)
        img = data['image']
        mask = data['mask']

        # Force resize to (256, 256) to fix dimension mismatches during batching
        img = F.interpolate(img.unsqueeze(0), size=(256, 256), mode='bilinear', align_corners=False).squeeze(0)

        # Our loss function and evaluation expect the mask to be (H, W) integer class indices (0 or 1).
        mask = F.interpolate(mask.unsqueeze(0), size=(256, 256), mode='nearest').squeeze(0).squeeze(0).long()

        return img, mask

# 1. Initialize the dataset using the directory we saved the tensors to
processed_dir = '/content/drive/MyDrive/INbreast_Processed'
full_dataset = ProcessedINbreastDataset(processed_dir)

# Ensure we have data
if len(full_dataset) == 0:
    print(f"Warning: No processed files found in {processed_dir}. Please check step 8.")
else:
    print(f"Total processed samples found: {len(full_dataset)}")

    # 2. Split into Train (80%) and Validation (20%)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    # Use a fixed generator seed for reproducible splits
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

    # 3. Create DataLoaders
    batch_size = 4
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f"Training set: {len(train_dataset)} samples ({len(train_loader)} batches)")
    print(f"Validation set: {len(val_dataset)} samples ({len(val_loader)} batches)")
    print("DataLoaders are ready for training!")

In [ ]:
import os

# Directory to save model checkpoints in Google Drive
checkpoint_dir = '/content/drive/MyDrive/INbreast_Project/Checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

def train_and_validate(model, train_loader, val_loader, optimizer, criterion, epochs, device):
    best_dice = 0.0
    history = {'train_loss': [], 'val_dice': [], 'val_iou': []}

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch+1}/{epochs} ---")

        # --- Training Phase ---
        model.train()
        train_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images) # outputs shape: (B, 2, 256, 256)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader) if len(train_loader) > 0 else 0
        print(f"Training Loss: {avg_train_loss:.4f}")
        history['train_loss'].append(avg_train_loss)

        # --- Validation Phase ---
        print("Running Validation...")
        avg_dice, avg_iou = evaluate_model(model, val_loader, device)
        history['val_dice'].append(avg_dice)
        history['val_iou'].append(avg_iou)

        # --- Model Checkpointing ---
        if avg_dice > best_dice:
            print(f"Validation Dice improved from {best_dice:.4f} to {avg_dice:.4f}. Saving model...")
            best_dice = avg_dice
            checkpoint_path = os.path.join(checkpoint_dir, 'best_transunet_model.pth')
            torch.save(model.state_dict(), checkpoint_path)

    print("\nTraining complete. Best model saved to Drive!")
    return best_dice, history

print("Full training loop with checkpointing is ready!")

In [ ]:
import torch.nn as nn
import torch.optim as optim

# --- 1. Combine Encoder and Decoder ---
class TransUNet(nn.Module):
    def __init__(self, encoder, decoder):
        super(TransUNet, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        features = self.encoder(x)
        out = self.decoder(features)
        return out

# Initialize the full model
# (In practice, you would load the pre-trained weights into the encoder first)
full_model = TransUNet(encoder, decoder).to(device)

# --- 2. Fine-Tuning Setup ---
criterion = nn.CrossEntropyLoss() # Assuming num_classes=2 (Background vs Tumor)
optimizer = optim.Adam(full_model.parameters(), lr=1e-4)

# --- 3. Fine-Tuning Loop Skeleton ---
num_finetune_epochs = 3
print("Starting Fine-Tuning phase...")

for epoch in range(num_finetune_epochs):
    full_model.train()
    epoch_loss = 0.0

    # Simulate a single batch of fine-tuning data
    # Shape: (Batch, Channels, H, W)
    images = torch.randn(4, 1, 256, 256).to(device)
    # Target masks for CrossEntropy should be (Batch, H, W) with class indices (0 or 1)
    masks = torch.randint(0, 2, (4, 256, 256)).long().to(device)

    optimizer.zero_grad()

    # Forward Pass
    outputs = full_model(images) # Shape: (B, 2, 256, 256)

    # Compute Segmentation Loss
    loss = criterion(outputs, masks)

    # Backward Pass & Optimize
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()
    print(f"Fine-Tune Epoch [{epoch+1}/{num_finetune_epochs}], Segmentation Loss: {epoch_loss:.4f}")

print("Fine-Tuning completed!")

In [ ]:
import matplotlib.pyplot as plt
import torch

# Ensure the Dense Contrastive Model is in evaluation mode
dense_model.eval()

print("Extracting a sample from the training set...")
# Get a batch from the training loader (since our val set lacked tumors)
train_images, train_masks = next(iter(train_loader))
sample_img = train_images[0:1].to(device) # Keep batch dimension: (1, 1, 256, 256)

with torch.no_grad():
    # Pass the image through the encoder and 1x1 projector to get the dense feature map
    # Output shape will be (Batch, Projection_Dim, H_feature, W_feature)
    dense_feature_map = dense_model(sample_img)

print(f"Input Image Shape: {sample_img.shape}")
print(f"Dense Feature Map Shape: {dense_feature_map.shape}")
print("Notice how spatial dimensions (32x32) are preserved, but we now have 128 channels of features per 'pixel'!")

# Visualize the original image and the first 3 channels of the dense feature map
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Plot Original Image
axes[0].imshow(sample_img.cpu().squeeze().numpy(), cmap='gray')
axes[0].set_title('Original Input Image (256x256)')
axes[0].axis('off')

# Plot the first 3 channels of the dense feature map
for i in range(3):
    # Extract a single channel from the dense feature map
    feature_channel = dense_feature_map[0, i, :, :].cpu().numpy()
    axes[i+1].imshow(feature_channel, cmap='viridis')
    axes[i+1].set_title(f'Dense Feature Map\nChannel {i+1} (32x32)')
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

### Understanding Dice Score and IoU

To calculate these metrics manually, break down the pixel predictions into three categories (ignoring True Negatives for these specific metrics):

*   **True Positives (TP):** The model predicted 'Tumor' (1), and the ground truth is 'Tumor' (1).
*   **False Positives (FP):** The model predicted 'Tumor' (1), but the ground truth is 'Background' (0).
*   **False Negatives (FN):** The model predicted 'Background' (0), but the ground truth is 'Tumor' (1).

#### 1. Intersection over Union (IoU) / Jaccard Index
IoU measures the exact overlap between the predicted mask and the true mask, divided by the total area covered by *both* masks combined.
*   **Formula:** `IoU = TP / (TP + FP + FN)`

#### 2. Dice Score / F1 Score
The Dice Score is very similar to IoU but gives exactly twice the weight to the True Positives. It is the most common metric in medical imaging.
*   **Formula:** `Dice = (2 * TP) / (2 * TP + FP + FN)`

*Note: We usually add a tiny number (like `1e-6`, called epsilon) to both the numerator and denominator to prevent a "Division by Zero" error when an image has no tumors.*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# 1. Ensure the whole model (Encoder + Decoder) is unfrozen
for param in full_model.parameters():
    param.requires_grad = True

print("All layers in the Encoder and Decoder are unfrozen and ready for fine-tuning.")

# 2. Define a combined Dice Loss + Cross Entropy Loss to handle severe class imbalance
class DiceCELoss(nn.Module):
    def __init__(self, weight_ce=0.2, weight_dice=2.0):
        super(DiceCELoss, self).__init__()
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
        self.ce = nn.CrossEntropyLoss()

    def forward(self, inputs, targets):
        # Cross Entropy part
        ce_loss = self.ce(inputs, targets)

        # Dice part
        # Inputs are raw logits: apply softmax to get probabilities for the positive class (tumor = index 1)
        probs = F.softmax(inputs, dim=1)[:, 1, :, :]

        # Flatten predictions and targets
        probs_flat = probs.contiguous().view(-1)
        targets_flat = targets.contiguous().view(-1).float()

        intersection = (probs_flat * targets_flat).sum()
        dice_loss = 1 - ((2. * intersection + 1e-6) / (probs_flat.sum() + targets_flat.sum() + 1e-6))

        return self.weight_ce * ce_loss + self.weight_dice * dice_loss

# Adjusted weights: Focus heavily on Dice loss to penalize False Negatives
segmentation_criterion = DiceCELoss(weight_ce=0.2, weight_dice=2.0)
print("Initialized combined Dice + CrossEntropy Loss with updated weights.")

# 3. Define the Optimizer
# Using AdamW with weight decay for better regularization, and a slightly lower learning rate
optimizer = optim.AdamW(full_model.parameters(), lr=5e-5, weight_decay=1e-4)

# 4. Run the Fine-Tuning Loop
fine_tune_epochs = 100
print(f"Starting Full Model Fine-Tuning for {fine_tune_epochs} epochs...")

# Using the train_and_validate function we defined earlier
best_val_dice, history = train_and_validate(
    model=full_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=segmentation_criterion,
    epochs=fine_tune_epochs,
    device=device
)

print(f"\nFine-tuning completed! Best Validation Dice: {best_val_dice:.4f}")


### Quantitative Evaluation: Confusion Matrix & Metrics
Let's calculate the overall performance on the validation set pixel-by-pixel.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import os
import torch

def evaluate_quantitatively(model, loader, device):
    model.eval()
    all_preds = []
    all_masks = []

    print("Running quantitative evaluation on the validation set...")
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            outputs = model(images)

            # Apply softmax/argmax to get class predictions
            preds = torch.argmax(outputs, dim=1).cpu().numpy().flatten()
            masks = masks.numpy().flatten()

            all_preds.extend(preds)
            all_masks.extend(masks)

    # Calculate Confusion Matrix
    cm = confusion_matrix(all_masks, all_preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    # Plot Confusion Matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Background (0)', 'Tumor (1)'],
                yticklabels=['Background (0)', 'Tumor (1)'])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Pixel-level Confusion Matrix')
    plt.show()

    # Calculate Metrics
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)

    print(f"\nEvaluation Results:")
    print(f"True Positives (Tumor pixels correctly identified): {tp}")
    print(f"False Positives (Background labeled as Tumor): {fp}")
    print(f"False Negatives (Tumor missed by model): {fn}")
    print(f"Overall Dice Score: {dice:.4f}")
    print(f"Overall IoU: {iou:.4f}")

# Load the best model weights before evaluating
best_checkpoint_path = os.path.join(checkpoint_dir, 'best_transunet_model.pth')
if os.path.exists(best_checkpoint_path):
    full_model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
    print(f"Loaded best model weights from {best_checkpoint_path}")
else:
    print("Best model checkpoint not found, evaluating with current weights.")

evaluate_quantitatively(full_model, val_loader, device)

### Qualitative Evaluation: Visualizing Predictions
Let's visually compare the model's output to the original images and ground truth masks.

In [ ]:
def visualize_predictions(model, loader, device, num_samples=3):
    model.eval()
    samples_shown = 0

    plt.figure(figsize=(15, 5 * num_samples))

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu()

            for i in range(images.size(0)):
                # Prioritize showing samples that actually contain a tumor in ground truth
                if masks[i].sum() > 0 or samples_shown == 0:
                    # Original Image
                    plt.subplot(num_samples, 3, samples_shown * 3 + 1)
                    plt.imshow(images[i].cpu().squeeze(), cmap='gray')
                    plt.title(f'Original Image {samples_shown+1}')
                    plt.axis('off')

                    # Ground Truth Mask
                    plt.subplot(num_samples, 3, samples_shown * 3 + 2)
                    plt.imshow(masks[i].squeeze(), cmap='gray')
                    plt.title('Ground Truth Mask')
                    plt.axis('off')

                    # Predicted Mask
                    plt.subplot(num_samples, 3, samples_shown * 3 + 3)
                    plt.imshow(preds[i].squeeze(), cmap='gray')
                    plt.title('Predicted Mask')
                    plt.axis('off')

                    samples_shown += 1
                    if samples_shown >= num_samples:
                        plt.tight_layout()
                        plt.show()
                        return

print("Visualizing predictions on the validation set...")
visualize_predictions(full_model, val_loader, device, num_samples=3)

### Tackling Severe Imbalance with Focal Loss
Standard Cross Entropy gets overwhelmed by the sheer number of "Background" pixels. **Focal Loss** down-weights the loss assigned to well-classified examples (easy background pixels) and focuses on hard examples (tumor pixels and boundaries). We combine this with **Dice Loss** to maintain spatial overlap quality.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs shape: (B, C, H, W)
        # targets shape: (B, H, W)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class FocalDiceLoss(nn.Module):
    def __init__(self, weight_focal=1.0, weight_dice=2.0, gamma=3.0):
        super(FocalDiceLoss, self).__init__()
        # High gamma (3.0) forces intense focus on hard-to-classify pixels
        self.focal = FocalLoss(gamma=gamma)
        self.weight_focal = weight_focal
        self.weight_dice = weight_dice

    def forward(self, inputs, targets):
        # Focal Loss component
        f_loss = self.focal(inputs, targets)

        # Dice Loss component (focuses on positive class index 1)
        probs = F.softmax(inputs, dim=1)[:, 1, :, :]
        probs_flat = probs.contiguous().view(-1)
        targets_flat = targets.contiguous().view(-1).float()

        intersection = (probs_flat * targets_flat).sum()
        d_loss = 1 - ((2. * intersection + 1e-6) / (probs_flat.sum() + targets_flat.sum() + 1e-6))

        return self.weight_focal * f_loss + self.weight_dice * d_loss

# 1. Initialize the new Loss Function
advanced_criterion = FocalDiceLoss(weight_focal=1.0, weight_dice=2.0, gamma=3.0)
print("Focal + Dice Loss Initialized!")

# 2. Reset the Model (Reload Pre-trained Encoder to start fresh)
# This prevents the model from being stuck in the local minima it fell into previously
print("Re-initializing the model to start a fresh fine-tuning run...")
fresh_encoder = TransUNetEncoder(in_channels=1, hidden_dim=512).to(device)
if os.path.exists(encoder_checkpoint_path):
    fresh_encoder.load_state_dict(torch.load(encoder_checkpoint_path, map_location=device))
    print("Pre-trained encoder weights reloaded.")

fresh_decoder = TransUNetDecoder(in_channels=512, num_classes=2).to(device)
fresh_full_model = TransUNet(fresh_encoder, fresh_decoder).to(device)

for param in fresh_full_model.parameters():
    param.requires_grad = True

# 3. Setup Optimizer (Slightly lower LR since Focal loss scales differently)
fresh_optimizer = optim.AdamW(fresh_full_model.parameters(), lr=3e-5, weight_decay=1e-4)
print("Ready for a new training run!")

In [ ]:
# Run Fine-Tuning with Focal Loss
num_focal_epochs = 50
print(f"Starting Fine-Tuning with Focal + Dice Loss for {num_focal_epochs} epochs...")

best_focal_dice, focal_history = train_and_validate(
    model=fresh_full_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=fresh_optimizer,
    criterion=advanced_criterion,
    epochs=num_focal_epochs,
    device=device
)

print(f"\nNew training completed! Best Validation Dice: {best_focal_dice:.4f}")

### Training History Comparison
Let's visualize the training loss, Validation Dice score, and Validation IoU for both the initial 100-epoch fine-tuning run (Dice + CrossEntropy) and the subsequent 50-epoch run (Focal + Dice Loss) to see how the model's learning dynamics changed.

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Training Loss Comparison
axes[0].plot(range(1, len(history['train_loss']) + 1), history['train_loss'], label='Run 1: Dice+CE (100 Epochs)', color='blue', alpha=0.7)
axes[0].plot(range(1, len(focal_history['train_loss']) + 1), focal_history['train_loss'], label='Run 2: Focal+Dice (50 Epochs)', color='red', alpha=0.7)
axes[0].set_title('Training Loss Comparison')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. Validation Dice Score Comparison
axes[1].plot(range(1, len(history['val_dice']) + 1), history['val_dice'], label='Run 1: Dice+CE', color='blue', alpha=0.7)
axes[1].plot(range(1, len(focal_history['val_dice']) + 1), focal_history['val_dice'], label='Run 2: Focal+Dice', color='red', alpha=0.7)
axes[1].set_title('Validation Dice Score Comparison')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Dice Score')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

# 3. Validation IoU Comparison
axes[2].plot(range(1, len(history['val_iou']) + 1), history['val_iou'], label='Run 1: Dice+CE', color='blue', alpha=0.7)
axes[2].plot(range(1, len(focal_history['val_iou']) + 1), focal_history['val_iou'], label='Run 2: Focal+Dice', color='red', alpha=0.7)
axes[2].set_title('Validation IoU Comparison')
axes[2].set_xlabel('Epochs')
axes[2].set_ylabel('IoU Score')
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Evaluating the Focal + Dice Loss Model
Let's evaluate our newly trained model to see if the Focal Loss successfully penalized the False Negatives and improved the boundaries.

In [ ]:
import os
import torch

# Load the best model weights from the focal loss run
best_checkpoint_path = os.path.join(checkpoint_dir, 'best_transunet_model.pth')
if os.path.exists(best_checkpoint_path):
    fresh_full_model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
    print(f"Loaded best focal loss model weights from {best_checkpoint_path}")
else:
    print("Best model checkpoint not found, evaluating with current weights.")

print("\n--- Quantitative Evaluation (Focal + Dice Loss) ---")
evaluate_quantitatively(fresh_full_model, val_loader, device)


In [ ]:
print("--- Qualitative Evaluation (Focal + Dice Loss) ---")
visualize_predictions(fresh_full_model, val_loader, device, num_samples=3)


# Report: TransUNet and Pixel-Level Contrastive Learning for Medical Image Segmentation

## 1. What the Topic Is
The topic explores the integration of **TransUNet** (a hybrid Vision Transformer and U-Net architecture) combined with **Pixel-Level Contrastive Learning** and **Focal Loss** to perform precise medical image segmentation. Specifically, this approach is applied to the INbreast dataset to segment breast cancer lesions, which often present with fuzzy and ill-defined boundaries.

## 2. What Problem Was It Designed to Address
Medical image segmentation, particularly for tumors in mammography, faces several critical challenges:
1. **Fuzzy Boundaries:** Low contrast between dense breast tissue and tumors makes it difficult for standard Convolutional Neural Networks (CNNs) to accurately delineate boundaries because they rely heavily on local pixel neighborhoods (limited receptive fields).
2. **Lack of Global Context:** Standard U-Nets struggle to understand the global anatomical structure of the breast, which is crucial for distinguishing a benign density from a malignant mass.
3. **Severe Class Imbalance:** In a typical mammogram, tumor pixels represent a tiny fraction (< 5%) of the total image, while the background and healthy tissue dominate. This biases traditional loss functions (like standard Cross-Entropy) toward predicting only the background.

## 3. Who Introduced It, When, and Where
The solution is a culmination of several pivotal breakthroughs in deep learning:
*   **U-Net:** Introduced by **Olaf Ronneberger, Philipp Fischer, and Thomas Brox** in **2015** at the **MICCAI** conference. It established the standard encoder-decoder architecture with skip connections for biomedical image segmentation.
*   **Vision Transformer (ViT):** Introduced by **Alexey Dosovitskiy et al. (Google Brain)** at **ICLR 2021**. It demonstrated that treating an image as a sequence of patches and processing them with pure self-attention (Transformers) captures excellent global context.
*   **TransUNet:** Introduced by **Jieneng Chen et al.** in **2021** (arXiv/CVPR). It bridged ViT and U-Net, using a CNN-Transformer hybrid encoder to capture both high-resolution local features and rich global context.
*   **Contrastive Learning (SimCLR / InfoNCE):** Popularized by **Ting Chen et al. (Google Brain)** at **ICML 2020**. It established the framework for learning representations by maximizing agreement between differently augmented views of the same data point via the NT-Xent loss.

## 4. How It Works (Theoretical & Mathematical Foundation)

### A. TransUNet Architecture
TransUNet first passes the image through a CNN feature extractor to get a feature map. This map is then divided into patches, flattened, and projected into sequence embeddings. These sequence embeddings are fed into a standard Transformer Encoder (comprising Multi-Head Self-Attention layers), which models long-range dependencies. Finally, the sequence is reshaped back into a spatial feature map and passed to a cascaded U-Net decoder, which progressively upsamples the features while fusing them with high-resolution CNN features via skip connections.

### B. Contrastive Learning & NT-Xent Loss
Contrastive learning forces the model to learn features such that similar samples (positive pairs) are close in the latent space, and dissimilar samples (negative pairs) are far apart.
The **Normalized Temperature-scaled Cross Entropy (NT-Xent) Loss** for a positive pair $(z_i, z_j)$ is defined as:

$$ \ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)} $$

Where:
*   $\text{sim}(u, v) = \frac{u^T v}{\|u\| \|v\|}$ is the cosine similarity.
*   $\tau$ is a temperature parameter that scales the distribution.
*   $N$ is the batch size (so there are $2N$ augmented views, providing $2N-1$ negatives).

In our *pixel-level* adaptation, instead of pulling whole images together, we pull dense feature representations of the *same spatial pixel* from two augmented views together, explicitly forcing the model to learn robust tissue boundary features invariant to spatial distortions.

### C. Focal Loss for Imbalance
To address the extreme background-to-tumor imbalance, we replaced standard Cross-Entropy with **Focal Loss** (Lin et al., ICCV 2017). It dynamically scales the cross-entropy loss based on prediction confidence:

$$ \text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t) $$

Where $\gamma$ (focusing parameter) reduces the relative loss for well-classified examples (easy background pixels), putting more focus on hard, misclassified examples (tumor boundaries).

## 5. Implementation in PyTorch
Below are the core PyTorch implementations utilized in our experimental setup.

**1. The NT-Xent Contrastive Loss:**
```python
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        z = torch.cat((z_i, z_j), dim=0)
        sim_matrix = torch.matmul(z, z.T) / self.temperature
        
        labels = torch.cat([torch.arange(batch_size) + batch_size, torch.arange(batch_size)], dim=0).to(z.device)
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
        sim_matrix.masked_fill_(mask, -9e15)
        
        return self.criterion(sim_matrix, labels)
```

**2. Focal + Dice Loss:**
```python
class FocalDiceLoss(nn.Module):
    def __init__(self, weight_focal=1.0, weight_dice=2.0, gamma=3.0):
        super(FocalDiceLoss, self).__init__()
        self.focal = FocalLoss(gamma=gamma) # Custom FocalLoss implementation
        self.weight_focal = weight_focal
        self.weight_dice = weight_dice

    def forward(self, inputs, targets):
        f_loss = self.focal(inputs, targets)
        probs = F.softmax(inputs, dim=1)[:, 1, :, :]
        probs_flat = probs.contiguous().view(-1)
        targets_flat = targets.contiguous().view(-1).float()
        
        intersection = (probs_flat * targets_flat).sum()
        d_loss = 1 - ((2. * intersection + 1e-6) / (probs_flat.sum() + targets_flat.sum() + 1e-6))
        return self.weight_focal * f_loss + self.weight_dice * d_loss
```

## 6. Experimental Setup and Results
Our experiment utilized the INbreast dataset, preprocessing full-resolution mammograms into 256x256 tensors. We executed two primary fine-tuning phases on our TransUNet model:

1.  **Phase 1 (Dice + CrossEntropy):** Over 100 epochs, the model achieved a best Validation Dice score of ~0.4285. However, quantitative evaluation revealed that standard CE loss favored predicting background, resulting in a staggering 12,653 False Negatives (missed tumor pixels) versus only 598 True Positives.
2.  **Phase 2 (Focal + Dice Loss):** By re-initializing the model and applying a highly weighted Focal Loss ($\gamma=3.0$), we explicitly penalized the model for missing difficult tumor pixels. Over 50 epochs, this shifted the learning dynamics. The quantitative results showed an increase in True Positives (1,462) and low False Positives (300). The False Negatives dropped to 11,789.

**Conclusion:** While Focal Loss successfully forced the network to identify more tumor structure (evidenced by the >2x increase in True Positives), the absolute number of False Negatives remains high, resulting in an overall Dice of ~0.1948 in the final evaluation. This highlights that while Contrastive Learning and TransUNet provide powerful theoretical advantages for context, severe medical image imbalance requires highly nuanced thresholding, patching strategies, or cascaded detection mechanisms to fully eradicate False Negatives.